# Phase 3: Review + Tips Embeddings (Week 9)

Generate Sentence-BERT embeddings for review text + tips and aggregate to user/destination profiles.

Using both reviews and tips provides richer semantic information about what users and destinations are like.

In [7]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from config import *
from src.data.cache import DataCache
from src.data.loader import load_reviews, load_tips
from src.models.embedding_utils import ReviewTipsEmbedder

cache = DataCache(str(CACHE_DIR))
businesses = cache.load_dataframe('businesses_major_cities')
reviews = cache.load_dataframe('reviews_cross_city')

## Load Tips Data

In [8]:
# Load tips (sample 10% same as reviews)
from config import RAW_DATA_DIR, PHASE1_SAMPLE_FRACTION
tips_file = RAW_DATA_DIR / 'yelp_academic_dataset_tip.json'

tips = load_tips(str(tips_file), sample=True, sample_fraction=PHASE1_SAMPLE_FRACTION)
print(f"Loaded {len(tips):,} tips")
print(f"Tips sample:")
print(tips.head(2))

Loaded 90,782 tips
Tips sample:
                  user_id             business_id  \
0  NBN4MgHP9D3cw--SnauTkA  QoezRbYQncpRqyrLH6Iqjg   
1  YVBB9g23nuVJ0u44zK0pSA  jtri188kuhe_AuEOJ51U_A   

                                                text                 date  \
0  They have lots of good deserts and tasty cuban...  2013-02-05 18:35:10   
1  This is probably the best place in the cool Sp...  2016-11-22 22:14:58   

   compliment_count  
0                 0  
1                 0  


## Initialize Embedder

In [9]:
# Initialize embedder (downloads model on first run)
print("Loading Sentence-BERT model (first time: ~500MB download)...")
embedder = ReviewTipsEmbedder(model_name='all-MiniLM-L6-v2')
print(f"✓ Model loaded. Embedding dimension: 384")

Loading Sentence-BERT model (first time: ~500MB download)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Model loaded. Embedding dimension: 384


## Generate User Embeddings

In [10]:
# Generate embeddings for sample users
user_embeddings = {}
sample_users = reviews['user_id'].unique()[:100]  # Sample first 100 users

for i, user_id in enumerate(sample_users):
    if i % 20 == 0:
        print(f"  Processing user {i+1}/100...")

    embedding = embedder.aggregate_user_embeddings(user_id, reviews, tips, businesses)
    if embedding is not None:
        user_embeddings[user_id] = embedding

print(f"✓ Generated embeddings for {len(user_embeddings)} users")

  Processing user 1/100...
  Processing user 21/100...
✓ Generated embeddings for 23 users


## Generate Destination Embeddings

In [11]:
# Generate embeddings for each city
dest_embeddings = {}
cities = businesses['city'].unique()

for city in cities:
    embedding = embedder.aggregate_destination_embeddings(city, reviews, tips, businesses)
    if embedding is not None:
        dest_embeddings[city] = embedding

print(f"✓ Generated embeddings for {len(dest_embeddings)} cities")
print(f"Cities: {list(dest_embeddings.keys())[:5]}...")

✓ Generated embeddings for 39 cities
Cities: ['St. Petersburg', 'Nashville', 'Indianapolis', 'Philadelphia', 'Santa Barbara']...


## Cache Embeddings

In [12]:
# Cache embeddings
cache.save_embeddings(user_embeddings, 'user_embeddings')
cache.save_embeddings(dest_embeddings, 'destination_embeddings')

print("✓ Embeddings cached")
print("Ready for Phase 4: Machine Learning Models")

✓ Embeddings cached
Ready for Phase 4: Machine Learning Models
